[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brkrishna/TickerTruth/blob/main/notebooks/bse_nse_dual_listing_reconciliation.ipynb)

# BSE/NSE Dual-Listing Reconciliation

Most India equity datasets cover only one exchange. But many companies are listed on both NSE and BSE simultaneously — and when they disagree on a corporate action date, your backtest is silently wrong.

TickerTruth ships an **ISIN bridge** (`fact_exchange_security_map`) that:
- Identifies every security dual-listed on NSE and BSE
- Flags cases where NSE and BSE report *different ex-dates* for the same event
- Cross-validates adjustment factors across both feeds

This notebook shows you how to use it.

**Data source:** `sample_data/exchange_security_map.parquet` and `sample_data/bse_scrip_master.parquet`

In [ ]:
import pandas as pd
from pathlib import Path

_cwd = Path.cwd()
SAMPLE_DATA = (
    _cwd / "sample_data"
    if (_cwd / "sample_data").exists()
    else _cwd.parent / "sample_data"
)

bridge = pd.read_parquet(SAMPLE_DATA / "exchange_security_map.parquet")
bse_scrips = pd.read_parquet(SAMPLE_DATA / "bse_scrip_master.parquet")
nse_securities = pd.read_parquet(SAMPLE_DATA / "security_master.parquet")

print(f"Bridge rows:     {len(bridge):,}")
print(f"BSE scrips:      {len(bse_scrips):,}")
print(f"NSE securities:  {len(nse_securities):,}")
print(f"Columns: {bridge.columns.tolist()}")

## 1. Exchange coverage breakdown

The ISIN bridge classifies every security into three buckets:

| Flag | Meaning |
|---|---|
| Neither flag | Dual-listed — ISIN exists on both NSE and BSE |
| `is_nse_only` | Listed only on NSE |
| `is_bse_only` | Listed only on BSE — NSE-only datasets miss these entirely |

In [ ]:
dual = (~bridge["is_bse_only"] & ~bridge["is_nse_only"]).sum()
nse_only = bridge["is_nse_only"].sum()
bse_only = bridge["is_bse_only"].sum()
total = len(bridge)

print(f"Total unique ISINs:  {total:,}")
print(f"Dual-listed:         {dual:,}  ({dual / total:.1%})")
print(f"NSE-only:            {nse_only:,}  ({nse_only / total:.1%})")
print(f"BSE-only:            {bse_only:,}  ({bse_only / total:.1%})")
print()
print("BSE-only securities are invisible to any NSE-only dataset.")
print(f"That's {bse_only:,} securities your NSE-only data pipeline doesn't know about.")

## 2. Dual-listed securities

For dual-listed securities, you can join NSE symbol ↔ BSE scrip code via ISIN.
This is the stable, unambiguous join key — ticker symbols change, scrip codes get recycled,
but ISIN persists across the lifetime of the legal entity.

In [ ]:
dual_listed = bridge[~bridge["is_bse_only"] & ~bridge["is_nse_only"]][
    ["isin", "nse_symbol", "bse_scrip_code"]
].dropna()

print(f"Dual-listed securities: {len(dual_listed):,}")
print()
# Show a sample
print(dual_listed.head(10).to_string(index=False))

## 3. Corporate action date conflicts

This is the high-value signal. When NSE says a dividend ex-date is June 1 and BSE says June 10,
one of them is wrong — and if your dataset uses the wrong one, your momentum strategy fires
on the wrong day.

TickerTruth pre-computes these conflicts in `ca_date_conflict` and provides the full
discrepancy table as a separate file.

In [ ]:
try:
    conflicts = pd.read_parquet(SAMPLE_DATA / "ca_date_conflicts.parquet")
    print(f"Total CA date conflicts: {len(conflicts):,}")
    print("Severity breakdown:")
    print(conflicts["conflict_severity"].value_counts().to_string())
    print()
    # Show the worst ones
    worst = conflicts.nlargest(5, "date_diff_days")
    print("Top 5 by date difference:")
    print(
        worst[
            [
                "isin",
                "action_code",
                "nse_ex_date",
                "bse_ex_date",
                "date_diff_days",
                "conflict_severity",
            ]
        ].to_string(index=False)
    )
except FileNotFoundError:
    print("ca_date_conflicts.parquet not in this sample — included in paid tiers.")
    print("The bridge column 'ca_date_conflict' flags ISINs where conflicts exist:")
    if "ca_date_conflict" in bridge.columns:
        flagged = bridge[bridge["ca_date_conflict"]]
        print(f"  {len(flagged)} ISINs have at least one CA date conflict")

## 4. BSE-only scrips — the hidden universe

Thousands of SME and regional companies are listed only on BSE. If your portfolio or
index includes these and you're using NSE-only data, you have no corporate action history
for them at all.

In [ ]:
bse_only_isins = bridge[bridge["is_bse_only"]]["isin"].values

# Look up their names from the BSE scrip master
bse_only_scrips = bse_scrips[bse_scrips["isin"].isin(bse_only_isins)].copy()
active_bse_only = bse_scrips[
    bse_scrips["isin"].isin(bse_only_isins) & bse_scrips["active_flag"]
]

print(f"BSE-only scrips (all):    {len(bse_only_scrips):,}")
print(f"BSE-only scrips (active): {len(active_bse_only):,}")
print()
print("Sample of active BSE-only companies:")
print(
    active_bse_only[["scrip_code", "isin", "scrip_name", "listing_date"]]
    .head(10)
    .to_string(index=False)
)

## 5. Cross-validating adjustment factors

For dual-listed securities, TickerTruth computes adjustment factors from both feeds
independently, then cross-validates them. A discrepancy means one feed has a data error
or an event that the other missed.

This is a strong signal to audit before using either factor in production.

In [ ]:
try:
    discrepancies = pd.read_parquet(
        SAMPLE_DATA / "bse_nse_factor_discrepancies.parquet"
    )
    print(f"Factor discrepancies found: {len(discrepancies):,}")
    if len(discrepancies):
        print(
            f"Severity: {discrepancies['discrepancy_severity'].value_counts().to_dict()}"
        )
        print()
        print("Worst discrepancies (review before using in production):")
        print(
            discrepancies.nlargest(5, "factor_diff")[
                [
                    "isin",
                    "as_of_date",
                    "bse_total_factor",
                    "nse_total_factor",
                    "factor_diff",
                    "discrepancy_severity",
                ]
            ].to_string(index=False)
        )
    else:
        print(
            "No discrepancies — NSE and BSE feeds agree on all adjustment factors in this sample."
        )
except FileNotFoundError:
    print(
        "bse_nse_factor_discrepancies.parquet not in this sample — included in paid tiers."
    )

---
## Key takeaways

1. **BSE-only securities are invisible to NSE-only data** — thousands of active companies exist only on BSE.
2. **CA date conflicts between NSE and BSE are real** — they're a strong signal of a data quality issue in one feed.
3. **The ISIN bridge is the stable join key** — use ISIN, not ticker symbol or scrip code, to track entities across exchanges and over time.
4. **Cross-validating adjustment factors catches errors** — if NSE and BSE disagree on a split factor, audit before using either in a live system.

**Want the full dataset?** See [tickertruth.com/pricing.html](https://tickertruth.com/pricing.html)